# Imports

In [1]:
import os
import re
import sqlite3
import pandas as pd
import numpy as np
import sys

# Global Constants

Random seed to reproduce randomness.

In [2]:
RANDOM_SEED = 42

Features selected to be used in the models.

In [3]:
SELECTED_FEATURES = ['destination_port', 'flow_duration', 'total_foward_packets',
       'total_backward_packets', 'total_length_of_foward_packets',
       'total_length_of_backward_packets', 'foward_packet_length_max',
       'foward_packet_length_min', 'foward_packet_length_average',
       'foward_packet_length_std', 'backward_packet_length_max',
       'backward_packet_length_min', 'backward_packet_length_average',
       'backward_packet_length_std', 'flow_bytes/s', 'flow_packets/s',
       'flow_iat_average', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min',
       'foward_iat_total', 'foward_iat_average', 'foward_iat_std',
       'foward_iat_max', 'foward_iat_min', 'backward_iat_total',
       'backward_iat_average', 'backward_iat_std', 'backward_iat_max',
       'backward_iat_min', 'foward_psh_flags', 'backward_psh_flags',
       'foward_urg_flags', 'backward_urg_flags', 'foward_header_length',
       'backward_header_length', 'foward_packets/s', 'backward_packets/s',
       'min_packet_length', 'max_packet_length', 'packet_length_average',
       'packet_length_std', 'packet_length_variance', 'fin_flag_count',
       'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count',
       'urg_flag_count', 'cwe_flag_count', 'ece_flag_count', 'down/up_ratio',
       'average_packet_size', 'average_foward_segment_size',
       'average_backward_segment_size', 'foward_average_bytes/bulk', 'foward_average_packets/bulk',
       'foward_average_bulk_rate', 'backward_average_bytes/bulk',
       'backward_average_packets/bulk', 'backward_average_bulk_rate',
       'subflow_foward_packets', 'subflow_foward_bytes',
       'subflow_backward_packets', 'subflow_backward_bytes',
       'init_win_bytes_forward', 'init_win_bytes_backward', 'act_data_packet_foward',
       'min_segment_size_forward', 'active_average', 'active_std', 'active_max',
       'active_min', 'idle_average', 'idle_std', 'idle_max', 'idle_min', 'label']

# Treatment in Input Files

## Renaming Files

The function below renames files that contain two dots in the name to have only one, to avoid problems with the SQLite schema.

In [4]:
# Rename CSV files with multiple dots in their names.
def rename_csvs_multiple_points(root_directory):
    
    for root, dirs, files in os.walk(root_directory):
        for filename in files:
            # Only processes .csv files with multiple dots
            if filename.endswith('.csv') and filename.count('.') > 1:
                # Remove the .csv extension
                name_without_csv = filename[:-4]  # Remove ".csv"
                
                # Replace all dots with underscores
                clean_name = name_without_csv.replace('.', '_')
                
                # Add .csv back
                new_name = clean_name + '.csv'
                
                # Full paths
                old_path = os.path.join(root, filename)
                new_path = os.path.join(root, new_name)
                
                # Check if the new name already exists
                if os.path.exists(new_path):
                    print(f"⚠️  File {new_name} already exists, skipping {filename}")
                    continue
                
                # Rename the file
                try:
                    os.rename(old_path, new_path)
                    print(f"✓ Renamed: {filename} → {new_name}")
                except OSError as e:
                    print(f"✗ Error renaming {filename}: {e}")

In [5]:
rename_csvs_multiple_points("../data/csv/")

## Setting Labels

One of the datasets needs labels to be declared.

In [ ]:
"""
c = 0
while True:
    if c == 0:
        path = f'../data/csv/CIC IoT-IDAD Dataset 2024/Web-Based/sqlinjection/SqlInjection_pcap_Flow.csv'
    else:
        path = f'../data/csv/CIC IoT-IDAD Dataset 2024/Web-Based/Uploading_Adasdasdasdasdasttack'
    label = "Web Attack SQL Injection"

    df = pd.read_csv(path)
    df['Label'] = label
    print(df['Label'])
    df.to_csv(path, index=False)

    c+=1
"""

# Database Creation

This part of the code is responsible for performing some processing on the data and exporting the result to a .db file. The purpose of using a database is because working only with dataframes requires a lot of RAM memory, and is not so viable.

## Export Data in .csv to a .db File

The function below is used to replace one word with another.

In [7]:
# Replace word function
def replace_word(text, old_word, new_word):
    # Escape any special characters in old_word
    escaped_old = re.escape(old_word)
    
    # Regex that ensures old_word is isolated:
    # - Start of string or non-alphanumeric character before
    # - End of string or non-alphanumeric character after
    pattern = fr'(^|[^A-Za-z0-9]){escaped_old}($|[^A-Za-z0-9])'
    
    # Replace while maintaining the delimiters around
    def replace_match(match):
        return match.group(1) + new_word + match.group(2)
    
    # Apply the substitution
    result = re.sub(pattern, replace_match, text)
    
    # Handle cases where old_word is at the beginning or end of the string
    if text.startswith(old_word) and len(text) > len(old_word) and text[len(old_word)] not in 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789':
        result = new_word + text[len(old_word):]
    if text.endswith(old_word) and len(text) > len(old_word) and text[-len(old_word)-1] not in 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789':
        result = text[:-len(old_word)] + new_word
    
    return result

The function below filters the data, allowing only the chosen characteristics(columns).

In [8]:
# Standardize column names like the SELECTED_FEATURES pattern
def format_to_allowed_columns(input_list):
    formatted = list()
    
    for name in input_list:

        name = name.strip()
        
        name = name.replace(' ', '_')
        
        name = name.lower()
        
        name = replace_word(name, 'src', 'source')
        name = replace_word(name, 'dst', 'destination')
        name = replace_word(name, 'len', 'length')
        name = replace_word(name, 'avg', 'average')
        name = replace_word(name, 'mean', 'average')
        name = replace_word(name, 'pkt', 'packet')
        name = replace_word(name, 'pkts', 'packets')
        name = replace_word(name, 'seg', 'segment')
        name = replace_word(name, 'fwd', 'foward')
        name = replace_word(name, 'bwd', 'backward')
        name = replace_word(name, 'cwr', 'cwe')
        
        # Specific fixes for cases that are not automatically captured
        corrections = {
            'foward_header_length.1': 'foward_header_length',
            'foward_header_lengthgth.1': 'foward_header_length',
            'foward_average_bytes_bulk': 'foward_average_bytes/bulk',
            'foward_average_packets_bulk': 'foward_average_packets/bulk',
            'foward_average_bulk_rate': 'foward_average_bulk_rate',
            'backward_average_bytes_bulk': 'backward_average_bytes/bulk',
            'backward_average_packets_bulk': 'backward_average_packets/bulk',
            'backward_average_bulk_rate': 'backward_average_bulk_rate',
            'foward_act_data_packets': 'act_data_packet_foward',
            'fwd_act_data_packets': 'act_data_packet_foward',
            'forward_act_data_packets': 'act_data_packet_foward',
            'foward_act_data_pkts': 'act_data_packet_foward',
            'fwd_act_data_pkts': 'act_data_packet_foward',
            'forward_act_data_pkts': 'act_data_packet_foward',
            'cwr_flag_count': 'cwe_flag_count',
            'flow_bytes_s': 'flow_bytes/s',
            'flow_packets_s': 'flow_packets/s',
            'foward_packets_s': 'foward_packets/s',
            'backward_packets_s': 'backward_packets/s',
            'foward_iat_total': 'foward_iat_total',
            'backward_iat_total': 'backward_iat_total',
            'min_seg_size_forward': 'min_segment_size_forward',
            'init_win_bytes_forward': 'init_win_bytes_forward',
            'init_win_bytes_backward': 'init_win_bytes_backward',
            'active_mean': 'active_average',
            'active_std': 'active_std',
            'active_max': 'active_max',
            'active_min': 'active_min',
            'idle_mean': 'idle_average',
            'idle_std': 'idle_std',
            'idle_max': 'idle_max',
            'idle_min': 'idle_min',
            'foward_segment_size_average':'average_foward_segment_size',
            'backward_segment_size_average':'average_backward_segment_size',
            'total_length_of_foward_packet':'total_length_of_foward_packets',
            'total_length_of_backward_packet':'total_length_of_backward_packets',
            'packet_length_min':'min_packet_length',
            'packet_length_max':'max_packet_length',
            'foward_bytes_bulk_average': 'foward_average_bytes/bulk',
            'foward_packet_bulk_average': 'foward_average_packets/bulk',
            'foward_bulk_rate_average': 'foward_average_bulk_rate',
            'backward_bytes_bulk_average': 'backward_average_bytes/bulk',
            'backward_packet_bulk_average': 'backward_average_packets/bulk',
            'backward_bulk_rate_average': 'backward_average_bulk_rate',
            'foward_init_win_bytes': 'init_win_bytes_forward',
            'backward_init_win_bytes': 'init_win_bytes_backward',
            'foward_segment_size_min': 'min_seg_size_forward',
            'label': 'label',
            'attack_name': 'label'
        }
        if name in corrections:
            name = corrections[name]

        formatted.append(name)
    
    return formatted

The function below standardizes the name of the values ​​in the label column, which is the data classification column.

In [9]:
def standardize_labels(input_list):
    corrections = {
        # Benign
        'BENIGN': 'Benign',
        'Benign': 'Benign',
        'Benign Traffic': 'Benign',
        'Label': 'Unknown',  # Remove não significativo
        
        # Ataques DDoS
        'DDOS attack-LOIC-UDP': 'DDoS LOIC UDP',
        'DDoS attacks-HOIC': 'DDoS HOIC',
        'DDoS attacks-LOIC-HTTP': 'DDoS LOIC HTTP',
        'DDoS HTTP Flood': 'DDoS HTTP Flood',
        'DDoS TCP SYN Flood': 'DDoS SYN Flood',
        'DDoS ICMP Flood': 'DDoS ICMP Flood',
        'DDoS UDP Flood': 'DDoS UDP Flood',
        'WebDDoS': 'DoS HTTP Flood',
        'MQTT DDoS Publish Flood': 'DDoS MQTT Publish Flood',
        'DDoS ACK Fragmentation': 'DDoS ACK Fragmentation',
        'DDoS ICMP Fragmentation': 'DDoS ICMP Fragmentation',
        'DDoS PSHACK Flood': 'DDoS PSHACK Flood',
        'DDoS RSTFIN Flood': 'DDoS RSTFIN Flood',
        'Mirai ACK Flood': 'DDoS Mirai ACK Flood',
        'Mirai HTTP Flood': 'DDoS Mirai HTTP Flood',
        'Mirai UDP Flood': 'DDoS Mirai UDP Flood',
        'DrDoS_DNS': 'DDoS DNS Flood',
        'DrDoS_LDAP': 'DDoS LDAP',
        'DrDoS_MSSQL': 'DDoS MSSQL',
        'DrDoS_NetBIOS': 'DDoS NetBIOS',
        'DrDoS_NTP': 'DDoS NTP',
        'DrDoS_SNMP': 'DDoS SNMP',
        'DrDoS_SSDP': 'DDoS SSDP',
        'DrDoS_UDP': 'DDoS UDP Flood',
        'UDPLag': 'DDoS UDP Flood',
        'DDoS': 'DDoS',
        'DDOS attack-HOIC': 'DDoS HOIC',
        
        # Ataques DoS
        'DoS slowloris': 'DoS Slowloris',
        'DoS attacks-Slowloris': 'DoS Slowloris',
        'DoS Slowhttptest': 'DoS SlowHTTPTest',
        'DoS attacks-SlowHTTPTest': 'DoS SlowHTTPTest',
        'Heartbleed': 'DoS Heartbleed',
        'MQTT DoS Connect Flood': 'DoS MQTT Connect Flood',
        'MQTT DoS Publish Flood': 'DoS MQTT Publish Flood',
        'MQTT Malformed': 'DoS MQTT Malformed',
        'DoS attacks-Hulk': 'DoS Hulk',
        'DoS attacks-GoldenEye': 'DoS GoldenEye',
        
        # Web Attacks
        'Web Attack ï¿½ Brute Force': 'Web Attack Brute Force',
        'Web Attack  Brute Force': 'Web Attack Brute Force',
        'Web Attack ï¿½ XSS': 'Web Attack XSS',
        'Web Attack  XSS': 'Web Attack XSS',
        'Web Attack ï¿½ Sql Injection': 'Web Attack SQL Injection',
        'Web Attack  Sql Injection': 'Web Attack SQL Injection',
        'XSS': 'Web Attack XSS',
        'Brute Force -Web': 'Web Attack Brute Force',
        'Web Attack  Brute Force': 'Web Attack Brute Force',
        'Web Attack  XSS': 'Web Attack XSS',
        'Web Attack  Sql Injection': 'Web Attack SQL Injection',
        'Brute Force -XSS': 'Web Attack XSS',

        # Brute Force
        'Dictionary Brute Force': 'Brute Force Dictionary',
        'FTP-Patator': 'Brute Force FTP',
        'FTP-BruteForce': 'Brute Force FTP',
        'SSH-Patator': 'Brute Force SSH',
        'SSH-Bruteforce': 'Brute Force SSH',
        'Sparta SSH Brute Force': 'Brute Force SSH',
        'Telnet Brute Force': 'Brute Force Telnet',
        'MQTT Brute Force': 'Brute Force MQTT',
        'Mirai Host Brute Force': 'Brute Force Mirai',
        
        # Varredura/Reconhecimento
        'PortScan': 'Reconnaissance Port Scanning',
        'Port Scanning': 'Reconnaissance Port Scanning',
        'Scan Host Port': 'Reconnaissance Port Scanning',
        'Scan Port OS': 'Reconnaissance Port Scanning',
        'Scan UDP Attack': 'Reconnaissance Port Scanning',
        'Reconnaissance': 'Reconnaissance',
        'Recon Host Discovery': 'Reconnaissance Host Discovery',
        'Recon OS Scan': 'Reconnaissance OS Fingerprinting',
        'Recon Ping Sweep': 'Reconnaissance Ping Sweep',
        'Recon Vulnerability Scan': 'Reconnaissance Vulnerability Scanning',
        'OS Fingerprinting': 'Reconnaissance OS Fingerprinting',
        'Vulnerability Scanner': 'Reconnaissance Vulnerability Scanning',
        'Portmap': 'Reconnaissance Port Scanning',
        'Recon Port Scan': 'Reconnaissance Port Scanning',
        'Scan Aggressive': 'Reconnaissance Port Scanning',

        
        # Malware/Bot
        'Infilteration': 'Infiltration',
        
        # MitM
        'MITM ARP Spoofing': 'MITM ARP Spoofing',
        'MITM': 'MITM',
        'DNS Spoofing': 'MITM DNS Spoofing',
        'ARP Spoofing': 'MITM ARP Spoofing',
        
        # Floods
        'UDP': 'DoS UDP Flood',
        'ACK Flood': 'DoS ACK Flood',
        'SYN Flood': 'DoS SYN Flood',
        'Syn': 'DoS SYN Flood',
        
        # Others
        'Generic': 'Unknown',
        'Analysis': 'Unknown',
        'SQL Injection': 'SQL Injection',
        'Uploading Attack': 'File Upload',
        'Fuzzers': 'Fuzzers',
        'TFTP': 'TFTP Attack',
        'UDP-lag': 'UDP Lag',
        '': 'Unknown',
        ' ': 'Unknown',
    }
    
    formatted = []
    for item in input_list:
        # Ensures the default value is used if it is None
        if type(item) is not str:
            item = 'Unknown'
        
        # Removes invisible characters (ASCII < 32 or non-printable Unicode)
        item = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', item)

        # Replaces multiple spaces or tabs with a single space
        item = re.sub(r'\s+', ' ', item).strip()

        # Looks for a correction or uses the original value if not found
        standard = corrections.get(item, item)

        formatted.append(standard)
    
    return formatted

The function below merges the duplicate columns.

In [10]:
# Combines homonymous columns preserving the first non-null value.
def merge_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.columns.duplicated().any():
        df = df.T.groupby(level=0).first().T
    return df

The function below loads the .csv files and transforms them into a .db file.

In [11]:
"""
Recursively traverses root_path, reads all CSV files, and writes
all the data into a single table in an SQLite database (db_path). 
Parameters 

root_path : str
    Root directory where the search begins (subdirectories will also be traversed).
db_path : str
    Path to the SQLite file that will receive the data.
allowed_columns : list[str], optional
    List of allowed columns. If provided, all CSV files must contain these columns.
chunksize : int
    Number of rows per batch read (adjust according to available RAM).
table_name : str
    Name of the single table where all data will be stored.
""" 
def load_csvs_to_db(
    root_path: str,
    db_path: str = "../data/data.db",
    allowed_columns: list[str] | None = None,
    chunksize: int = 100_000,
    table_name: str = "packets_flow_features"
) -> None:
    conn = sqlite3.connect(db_path)
    
    try:
        # Remove old table if it already exists
        conn.execute(f"DROP TABLE IF EXISTS {table_name}")
        
        table_created = False
        
        for dirpath, _, filenames in os.walk(root_path):
            for fname in filenames:
                if not fname.lower().endswith(".csv"):
                    continue

                csv_path = os.path.join(dirpath, fname)
                print(f"Importing {csv_path} → tabela '{table_name}'")

                for chunk in pd.read_csv(csv_path, 
                                         chunksize=chunksize,
                                         encoding="latin1",
                                         low_memory=False):
                    
                    # 1) Normalize the chunk header and standardize labels
                    chunk.columns = format_to_allowed_columns(chunk.columns)
                    chunk = merge_duplicate_columns(chunk)
                    chunk['label'] = standardize_labels(chunk['label'])
                    
                    # 2) Validate/adjust columns as allowed columns
                    if allowed_columns is not None:
                        # Add missing columns with NaN values
                        for col in allowed_columns:
                            if col not in chunk.columns:
                                chunk[col] = np.nan
                        # Keeps only the desired columns, in the specified order
                        chunk = chunk[allowed_columns]
                    else:
                        # If allowed_columns was not provided, set it dynamically
                        if not table_created:
                            allowed_columns = format_to_allowed_columns(chunk.columns.tolist())
                            print(f"Using dynamic columns: {allowed_columns}")
                    
                    # 3) Creates the table in the first iteration (with defined schema)
                    if not table_created:
                        # Creates the table with normalized columns
                        columns_sql = ", ".join([f'"{col}" TEXT' for col in chunk.columns])
                        conn.execute(f'CREATE TABLE "{table_name}" ({columns_sql})')
                        table_created = True
                    
                    # 4) Save to the bank
                    chunk.to_sql(table_name, conn, if_exists="append", index=False)

                print(f"✓ Completed: {csv_path}")

    except Exception as e:
        exc_type, exc_obj, exc_tb = sys.exc_info()
        print(f"❌ Error during execution: {e}, function: {exc_tb.tb_frame.f_code.co_name}, file: {exc_tb.tb_frame.f_code.co_filename}, line: {exc_tb.tb_lineno}")
    finally:
        conn.close()
        print("Database connection closed.")

Database file path declaration.

In [1]:
db_path = "../data/data.db"

If the database file does not exist, one is created.

In [ ]:
if not os.path.exists(db_path):
    load_csvs_to_db("../data/", db_path, SELECTED_FEATURES)
else:
    print(f"O banco de dados '{db_path}' já existe. Pulando carregamento.")